<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/toy_model/Analyse-Failures-Best_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
!pip install transformer_lens

In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm

E = 100  # num entities
T = 10   # num types/relations

SEP = E + T
Q = E + T + 1
PAD = E + T + 2
D_VOCAB = E + T + 3

ENTITIES = np.arange(0, E)
TYPES    = np.arange(E, E + T)

N_WORLDS = 80_000
MIN_FACTS, MAX_FACTS = 4, 8
SEED = 0

rng = np.random.default_rng(SEED)

def produce_example(num_relations: int, *, allow_self_loops: bool = False):
    facts = []
    seen_head_rel = set()

    while len(facts) < num_relations:
        e = int(rng.integers(0, E))
        t = int(TYPES[rng.integers(0, T)])

        if (e, t) in seen_head_rel:
            continue  # reject duplicate (e, t) - because then our question (t, e) would no longer be deterministic

        # forbid self-loop (can discuss?)
        e2 = int(rng.integers(0, E))
        while e2 == e:
            e2 = int(rng.integers(0, E))

        seen_head_rel.add((e, t))
        facts.append((e, t, e2))

    # choose a query by selecting one of the world facts
    q_idx = int(rng.integers(0, num_relations))
    Eq, Tq, E2q = facts[q_idx]

    seq = []
    for (e, t, e2) in facts:
        seq.extend([e, t, e2, SEP])
    seq.extend([Tq, Eq, Q])

    label = E2q
    return seq, label, facts

# --- Build dataset ---
rows = []
for _ in tqdm(range(N_WORLDS)):
    k = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))
    seq, label, _ = produce_example(k, allow_self_loops=False)
    rows.append({"tokens": seq, "label": label})

df = pd.DataFrame(rows)

100%|██████████| 80000/80000 [00:10<00:00, 7346.60it/s]


In [4]:
df

,tokens,label
0,"[63, 105, 26, 110, 30, 100, 7, 110, 1, 101, 81...",26
1,"[72, 108, 17, 110, 8, 108, 2, 110, 54, 100, 29...",52
2,"[46, 109, 80, 110, 98, 103, 68, 110, 95, 106, ...",38
3,"[52, 103, 31, 110, 42, 104, 71, 110, 88, 100, ...",71
4,"[62, 100, 8, 110, 37, 108, 40, 110, 78, 103, 2...",33
...,...,...
79995,"[26, 100, 22, 110, 7, 107, 13, 110, 21, 106, 6...",86
79996,"[71, 102, 9, 110, 87, 106, 54, 110, 43, 101, 6...",68
79997,"[32, 104, 40, 110, 22, 105, 96, 110, 97, 104, ...",40
79998,"[80, 105, 23, 110, 93, 105, 5, 110, 20, 101, 9...",23


In [5]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split

In [6]:
IGNORE_INDEX = -100

class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

In [7]:
def train_collate_fn(batch, rng=np.random.default_rng()):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = shuffle_facts(seq.tolist() if torch.is_tensor(seq) else seq, rng)
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target

def val_collate_fn(batch):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = seq.tolist() if torch.is_tensor(seq) else seq
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target


In [8]:
def shuffle_facts(seq, rng=np.random.default_rng()):
    seps = [i for i,t in enumerate(seq) if t == SEP]
    context = seq[:seps[-1]+1]
    query_part = seq[seps[-1]+1:]
    facts = [context[i:i+4] for i in range(0, len(context), 4)]
    rng.shuffle(facts)

    return [x for f in facts for x in f] + query_part

In [9]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_dataset = EntityBindingDataset(train_df)
val_dataset = EntityBindingDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=train_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=val_collate_fn)

# Save dataframes
train_df.to_csv("train_df.csv", index=False)
val_df.to_csv("val_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)

In [32]:
test_loader = DataLoader(EntityBindingDataset(test_df), batch_size=32, shuffle=False, collate_fn=val_collate_fn)

In [10]:
len(train_dataset)

64000

In [11]:
def compute_accuracy(logits: torch.Tensor, targets: torch.Tensor, ignore_index: int = IGNORE_INDEX):
    """
    Accuracy over positions where targets != ignore_index.
    Returns correct_count and total_count
    """
    with torch.no_grad():
        mask = targets.ne(ignore_index)
        total = mask.sum().item()
        preds = logits.argmax(dim=-1)
        correct = preds.masked_select(mask).eq(targets.masked_select(mask)).sum().item()
        return correct, total

In [21]:
import math
import itertools
import torch
import torch.nn as nn
from transformer_lens import HookedTransformer, HookedTransformerConfig
from torch.utils.tensorboard import SummaryWriter

# hparams

LAYERS = [3]
HEADS  = [2]

d_model = 256
d_mlp   = 1024
n_ctx   = 64
lr = 3e-4
betas = (0.9, 0.98)
weight_decay = 0.01
num_epochs = 30

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        d_mlp=d_mlp,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        act_fn="gelu",
        attn_only=True,
        normalization_type="LN",
    )
    return HookedTransformer(cfg)

In [22]:
model = build_model(3, 2).to("cuda")

Moving model to device:  cuda


In [23]:
pretrained_weights = torch.load("/content/D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt", map_location="cuda", weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

<All keys matched successfully>

In [28]:
def eval(model, loader):
    torch.manual_seed(0)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(0)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    total_correct = 0
    total_count = 0
    with torch.no_grad():
        for input_tokens, targets in loader:
            input_tokens, targets = input_tokens.to(device), targets.to(device)
            logits = model(input_tokens)
            c, n = compute_accuracy(logits, targets)
            total_correct += c
            total_count += n

    acc = (total_correct / total_count) if total_count else 0.0
    print(f"Acc: {acc:.10f}")

In [29]:
eval(model, train_loader)

Moving model to device:  cuda
Acc: 0.9996562500


In [30]:
eval(model, val_loader)

Moving model to device:  cuda
Acc: 0.9995000000


In [33]:
eval(model, test_loader)

Moving model to device:  cuda
Acc: 0.9997500000


In [34]:
for input, label in val_dataset:
  logits = model(input)
  sm = torch.softmax(logits.view(-1, logits.size(-1))[-1, :], dim=-1)
  top_probs, top_idxs = torch.topk(sm, k)

  if top_idxs[0].item() != label.item():
      print("input", input)
      print("label", label.item())
      for rank, (idx, p) in enumerate(zip(top_idxs.tolist(), top_probs.tolist()), start=1):
          print(f"  top{rank}: idx={idx}  prob={p:.6f}")

input tensor([ 37, 108,  65, 110,  94, 101,  10, 110,  66, 102,  38, 110,  90, 104,
         94, 110,  13, 109,   6, 110,   4, 104,  86, 110,  85, 108,  32, 110,
         94, 108,  84, 110, 108,  94, 111]) label tensor(84) pred 10 probability 0.6672084331512451
input tensor([  4, 109,  35, 110,  39, 109,  84, 110,  88, 109,  63, 110,  39, 104,
         45, 110, 109,  39, 111]) label tensor(84) pred 7 probability 0.20797523856163025
input tensor([ 44, 100,  91, 110,  11, 104,  54, 110,  81, 106,  40, 110,  37, 105,
         58, 110,  82, 100,  95, 110,  81, 100,   6, 110,  67, 109,  54, 110,
          8, 109,  23, 110, 100,  81, 111]) label tensor(6) pred 40 probability 0.9999871253967285
input tensor([ 75, 101,  72, 110,  94, 107,  60, 110,  31, 109,  38, 110,   7, 104,
         93, 110,  33, 100,  65, 110,   7, 101,  17, 110, 101,   7, 111]) label tensor(17) pred 93 probability 0.7462605237960815


In [35]:
for input, label in val_dataset:
  logits = model(input)
  sm = torch.softmax(logits.view(-1, logits.size(-1))[-1, :], dim=-1)
  top_probs, top_idxs = torch.topk(sm, k)

  if top_idxs[0].item() != label.item():
      print("input", input)
      print("label", label.item())
      for rank, (idx, p) in enumerate(zip(top_idxs.tolist(), top_probs.tolist()), start=1):
          print(f"  top{rank}: idx={idx}  prob={p:.6f}")

input tensor([ 37, 108,  65, 110,  94, 101,  10, 110,  66, 102,  38, 110,  90, 104,
         94, 110,  13, 109,   6, 110,   4, 104,  86, 110,  85, 108,  32, 110,
         94, 108,  84, 110, 108,  94, 111])
label 84
  top1: idx=10  prob=0.667208
  top2: idx=65  prob=0.180099
  top3: idx=84  prob=0.109359
  top4: idx=8  prob=0.004159
  top5: idx=76  prob=0.003158
input tensor([  4, 109,  35, 110,  39, 109,  84, 110,  88, 109,  63, 110,  39, 104,
         45, 110, 109,  39, 111])
label 84
  top1: idx=7  prob=0.207975
  top2: idx=84  prob=0.172206
  top3: idx=10  prob=0.095770
  top4: idx=35  prob=0.075343
  top5: idx=4  prob=0.069693
input tensor([ 44, 100,  91, 110,  11, 104,  54, 110,  81, 106,  40, 110,  37, 105,
         58, 110,  82, 100,  95, 110,  81, 100,   6, 110,  67, 109,  54, 110,
          8, 109,  23, 110, 100,  81, 111])
label 6
  top1: idx=40  prob=0.999987
  top2: idx=79  prob=0.000005
  top3: idx=10  prob=0.000003
  top4: idx=77  prob=0.000001
  top5: idx=43  prob=0.0000

In [ ]:
# Save the model
torch.save(model.state_dict(), "entity_binding_model.pth")

# Load the model
# Create a new model instance with the same configuration
loaded_model = HookedTransformer(cfg)
loaded_model.load_state_dict(torch.load("entity_binding_model.pth"))
loaded_model = loaded_model.to(device)

print("Model saved and loaded successfully.")

In [ ]:

loaded_model.eval()
total_val_loss = 0
total_val_correct = 0
total_val_count = 0
with torch.no_grad():
    for input_tokens, targets in val_loader:
        input_tokens, targets = input_tokens.to(device), targets.to(device)
        logits = loaded_model(input_tokens)
        loss = criterion(logits.view(-1, logits.size(-1)),
                      targets.view(-1))
        total_val_loss += loss.item()
        c, n = compute_accuracy(logits, targets)
        total_val_correct += c
        total_val_count += n

avg_val_loss = total_val_loss / len(val_loader)
val_acc = (total_val_correct / total_val_count) if total_val_count else 0.0

In [ ]:
val_acc, avg_val_loss